# Phase 0 — Lesson 09: Data Management
> Data is the fuel. How you manage it determines how fast you go.

**Source:** ai-engineering-from-scratch -> Phase 0 (Setup & Tooling), Lesson 09
**Type:** Build | **Language:** Python | **Time:** ~45 minutes
**Prerequisites:** Phase 0, Lesson 01

This lesson explains *why* data format and versioning choices matter, not just how to load a dataset. Run the code cells to see visualizations of the data pipeline and format tradeoffs.


# The Problem: Data Is the Bottleneck

---


Every AI project starts with data. You need to find datasets, download them, convert between formats, split them for training and evaluation, and version them so experiments are reproducible. Doing this manually every time is slow and error-prone. You need a repeatable workflow.


# Visual: The Data Pipeline — From Hub to Training

---


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(10, 4))
ax.axis('off')

steps = [
    ('Hugging Face Hub', 'Where datasets live', '#3498db'),
    ('datasets library', 'Download + cache', '#2ecc71'),
    ('Load / Stream', 'Full or row-by-row', '#9b59b6'),
    ('Local Cache', '~/.cache/huggingface/', '#e74c3c'),
    ('Format Conversion', 'CSV, JSON, Parquet, Arrow', '#f39c12'),
    ('Data Splits', 'train / val / test', '#e67e22'),
    ('Training Pipeline', 'Your model sees data', '#95a5a6'),
]

y = 0.92
for label, desc, color in steps:
    rect = patches.Rectangle((0.1, y-0.09), 0.8, 0.16, facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(0.15, y, label, ha='left', va='center', color='white', fontweight='bold', fontsize=10)
    ax.text(0.88, y, desc, ha='right', va='center', fontsize=8, style='italic')
    y -= 0.13

for i in range(len(steps)-1):
    ax.annotate('', xy=(0.5, y+0.02), xytext=(0.5, y+0.06),
                arrowprops=dict(arrowstyle='->', color='black', lw=2))
    y -= 0.13

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title("The Data Pipeline: Hub → Cache → Splits → Training", fontsize=13, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()


> **The Hugging Face `datasets` library** is the standard way to load data for AI work. It handles downloading, caching, format conversion, and streaming out of the box.


# Step 1: Install the datasets library

---


```bash
pip install datasets huggingface_hub
```


# Step 2: Load a Dataset

---


In [ ]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")
print(dataset)
print(dataset["train"][0])

# After first download, loads from cache at ~/.cache/huggingface/datasets/


This downloads the IMDB movie review dataset. After the first download, it loads from cache — no re-downloading needed.


# Step 3: Stream Large Datasets

---


Some datasets are too large to fit on disk. **Streaming** loads them row by row without downloading the full thing.

```python
dataset = load_dataset("wikimedia/wikipedia", "20231101.en", split="train", streaming=True)

for i, example in enumerate(dataset):
    print(example["title"])
    if i >= 4:
        break
```

Streaming gives you an `IterableDataset`. You process rows as they arrive. Memory usage stays constant regardless of dataset size.


# Visual: Streaming vs Full Download — Memory Usage

---


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Full load
ax = axes[0]
ax.axis('off')
rect = patches.Rectangle((0.1, 0.3), 0.8, 0.5, facecolor='#e74c3c', edgecolor='black', linewidth=2)
ax.add_patch(rect)
ax.text(0.5, 0.65, 'FULL DOWNLOAD', ha='center', va='center', color='white', fontweight='bold', fontsize=12)
ax.text(0.5, 0.5, 'Download entire dataset\nto disk first', ha='center', va='center', fontsize=9, color='white')
ax.text(0.5, 0.4, 'Memory grows with\ndataset size', ha='center', va='center', fontsize=9, color='white')
ax.text(0.5, 0.25, 'Fails if disk is full', ha='center', va='center', fontsize=9, color='white')

# Streaming
ax = axes[1]
ax.axis('off')
rect = patches.Rectangle((0.1, 0.3), 0.8, 0.5, facecolor='#27ae60', edgecolor='black', linewidth=2)
ax.add_patch(rect)
ax.text(0.5, 0.65, 'STREAMING', ha='center', va='center', color='white', fontweight='bold', fontsize=12)
ax.text(0.5, 0.5, 'Load row by row\non demand', ha='center', va='center', fontsize=9, color='white')
ax.text(0.5, 0.4, 'Constant memory\nregardless of size', ha='center', va='center', fontsize=9, color='white')
ax.text(0.5, 0.25, 'Works for\nterabyte datasets', ha='center', va='center', fontsize=9, color='white')

plt.tight_layout()
plt.show()


> **Use streaming** when datasets are too large for your disk. **Use full load** for smaller datasets where random access is needed.


# Step 4: Dataset Formats — CSV, JSON, Parquet, Arrow

---


The `datasets` library uses Apache Arrow under the hood. You can convert to other formats depending on what your pipeline needs.

```python
dataset = load_dataset("stanfordnlp/imdb", split="train")

dataset.to_csv("imdb_train.csv")
dataset.to_json("imdb_train.json")
dataset.to_parquet("imdb_train.parquet")
```

| Format | Size | Read Speed | Best For |
|--------|------|-----------|----------|
| CSV | Large | Slow | Human readability, spreadsheets |
| JSON | Large | Slow | APIs, nested data |
| Parquet | Small | Fast | Analytics, columnar queries |
| Arrow | Small | Fastest | In-memory processing (what `datasets` uses internally) |

For AI work, **Parquet is the best storage format**. Arrow is what you work with in memory. CSV and JSON are for interchange.


# Visual: Format Size vs Read Speed

---


In [ ]:
import matplotlib.pyplot as plt

formats = ['CSV', 'JSON', 'Parquet', 'Arrow']
size_mb = [500, 450, 80, 60]  # relative size
read_speed = [2, 2, 9, 10]  # relative speed

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Size
ax = axes[0]
ax.bar(formats, size_mb, color=['#e74c3c', '#e74c3c', '#27ae60', '#27ae60'], alpha=0.8)
ax.set_ylabel('Size (MB)', fontsize=10)
ax.set_title('File Size', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Read speed
ax = axes[1]
ax.bar(formats, read_speed, color=['#e74c3c', '#e74c3c', '#27ae60', '#27ae60'], alpha=0.8)
ax.set_ylabel('Read Speed (relative)', fontsize=10)
ax.set_title('Read Speed', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


> **Parquet** is columnar — it compresses better and reads specific columns faster. Use it for storing datasets you'll reuse.


# Step 5: Train / Validation / Test Splits

---


Every ML project needs three splits:

- **Train**: The model learns from this (typically 80%)
- **Validation**: You check progress during training (typically 10%)
- **Test**: Final evaluation after training is done (typically 10%)

Some datasets come pre-split. When they don't, split them yourself:

```python
dataset = load_dataset("stanfordnlp/imdb", split="train")

split = dataset.train_test_split(test_size=0.2, seed=42)
train_val = split["train"].train_test_split(test_size=0.125, seed=42)

train_ds = train_val["train"]
val_ds = train_val["test"]
test_ds = split["test"]

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")
```

**Always set a seed** for reproducibility. The same seed produces the same split every time.


# Visual: Train/Validation/Test Split with Fixed Seed

---


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.axis('off')

splits = [
    ('Train', 80, '#27ae60', 'Model learns from this'),
    ('Validation', 10, '#f39c12', 'Tune hyperparameters'),
    ('Test', 10, '#e74c3c', 'Final unbiased evaluation'),
]

sizes = [s[1] for s in splits]
labels = [f"{s[0]}\n{s[1]}%" for s in splits]
colors = [s[2] for s in splits]

wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, colors=colors, autopct='%1.0f%%',
    startangle=90, textprops={'fontsize': 10, 'fontweight': 'bold'}
)

for i, (name, pct, color, desc) in enumerate(splits):
    ax.text(1.3, 0.7 - i*0.15, f"{name}: {desc}", fontsize=9,
            color=color, fontweight='bold', va='center')

ax.set_title('Train/Validation/Test Split (80/10/10)', fontsize=12, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()


> **Never peek at the test set during training.** It's for final evaluation only. Use validation for hyperparameter tuning.


# Step 6: Download and Cache Models

---


Models are large files. The `huggingface_hub` library handles downloading and caching.

```python
from huggingface_hub import hf_hub_download, snapshot_download

model_path = hf_hub_download(
    repo_id="sentence-transformers/all-MiniLM-L6-v2",
    filename="config.json"
)
print(f"Cached at: {model_path}")

model_dir = snapshot_download("sentence-transformers/all-MiniLM-L6-v2")
print(f"Full model at: {model_dir}")
```

Models cache to `~/.cache/huggingface/hub/`. Once downloaded, they load instantly on subsequent runs.


# Step 7: Handle Large Files — .gitignore, Git LFS, or DVC

---


Model weights and large datasets should not go into git. Three options:

**Option A: `.gitignore` (simplest)**
```
*.bin
*.safetensors
*.pt
*.onnx
data/*.parquet
data/*.csv
models/
```

**Option B: Git LFS (track large files in git)**
```bash
git lfs install
git lfs track "*.bin"
git lfs track "*.safetensors"
git add .gitattributes
```

Git LFS stores pointers in your repo and the actual files on a separate server. GitHub gives you 1 GB free.

**Option C: DVC (data version control)**
```bash
pip install dvc
dvc init
dvc add data/training_set.parquet
git add data/training_set.parquet.dvc data/.gitignore
git commit -m "Track training data with DVC"
```

DVC creates small `.dvc` files that point to your data. The data itself lives in S3, GCS, or another remote storage backend.

| Approach | Complexity | Best For |
|----------|-----------|----------|
| `.gitignore` | Low | Personal projects, downloaded data you can re-fetch |
| Git LFS | Medium | Teams sharing model weights via git |
| DVC | High | Reproducible experiments, large datasets, teams |

For this course, `.gitignore` is enough. Use DVC when you need to reproduce exact experiments across machines.


# Visual: Large File Handling — Three Approaches

---


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

options = [
    ('.gitignore', 'Low\nPersonal projects\nRe-fetch data', '#27ae60'),
    ('Git LFS', 'Medium\nTeams sharing weights\n1 GB free on GitHub', '#f39c12'),
    ('DVC', 'High\nReproducible experiments\nS3 / GCS backend', '#e74c3c'),
]

for i, (name, desc, color) in enumerate(options):
    ax = axes[i]
    ax.axis('off')
    rect = plt.Rectangle((0.1, 0.2), 0.8, 0.6, facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(0.5, 0.7, name, ha='center', va='center', color='white', fontweight='bold', fontsize=13)
    ax.text(0.5, 0.45, desc, ha='center', va='center', fontsize=9, color='white')

plt.tight_layout()
plt.show()


> **Start with `.gitignore`.** Move to Git LFS or DVC only when you need to share or version large files.


# Step 8: Storage Patterns — Local vs Cloud

---


**Local storage** works for datasets under ~10 GB. The HF cache handles this automatically.

**Cloud storage** is for anything larger or shared across machines:

```python
import os

local_path = os.path.expanduser("~/.cache/huggingface/datasets/")

# s3_path = "s3://my-bucket/datasets/"
# gcs_path = "gs://my-bucket/datasets/"
```

DVC integrates with S3 and GCS directly:
```bash
dvc remote add -d myremote s3://my-bucket/dvc-store
dvc push
```

For this course, local storage is sufficient. Cloud storage becomes relevant when you fine-tune on remote GPU instances.


# Visual: Datasets Used in This Course

---


In [ ]:
import matplotlib.pyplot as plt

datasets = ['IMDB', 'WikiText', 'SQuAD', 'MNIST', 'Common Crawl', 'COCO']
lessons = [2, 2, 1, 1, 1, 1]
sizes_mb = [84, 181, 35, 21, 10000, 5000]  # approximate

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(datasets, sizes_mb, color='#3498db', alpha=0.8)
ax.set_ylabel('Size (MB)', fontsize=10)
ax.set_title('Datasets Used Across the Course', fontsize=12, fontweight='bold')
ax.set_yscale('log')
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=15, ha='right')

for bar, mb in zip(bars, sizes_mb):
    ax.text(bar.get_x() + bar.get_width()/2, mb * 1.1, f'{mb:,}', ha='center', fontweight='bold', fontsize=8)

plt.tight_layout()
plt.show()


| Dataset | Lessons | Size | What It Teaches |
|---------|---------|------|----------------|
| IMDB | Tokenization, classification | 84 MB | Text classification basics |
| WikiText | Language modeling | 181 MB | Next-token prediction |
| SQuAD | QA systems | 35 MB | Question answering, spans |
| Common Crawl (subset) | Embeddings | Varies | Large-scale text processing |
| MNIST | Vision basics | 21 MB | Image classification fundamentals |
| COCO (subset) | Multimodal | Varies | Image-text pairs |

You do not need to download all of these now. Each lesson specifies what it needs.


# Quiz Review

---


Try answering these before reading the explanations:

**Q1:** Why is it important to have separate train, validation, and test splits?
> **Explanation:** The training set teaches the model, the validation set tunes hyperparameters during training, and the test set provides a final unbiased evaluation on data the model has never seen.

**Q2:** What is the Hugging Face Hub primarily used for?
> **Explanation:** Hosting and sharing datasets, models, and ML artifacts. The `datasets` library loads data from it.

**Q3:** What advantage does Parquet have over CSV?
> **Explanation:** Parquet uses columnar storage for smaller file sizes and faster read speeds. It's the preferred storage format for ML datasets.

**Q4:** What does `streaming=True` do?
> **Explanation:** Loads data row by row without downloading the full dataset to disk. Memory usage stays constant regardless of dataset size.

**Q5:** When should you use DVC instead of `.gitignore`?
> **Explanation:** When you need to reproduce exact experiments across machines with versioned data. DVC tracks data versions with pointer files in git while storing actual data in remote storage.


# Practice Exercises

---


Do these in order:

**Exercise 1:** Load the `glue` dataset with the `mrpc` config and inspect the first 5 examples.

**Exercise 2:** Stream the `c4` dataset and count how many examples you can process in 10 seconds.

**Exercise 3:** Convert a dataset to Parquet and compare the file size to CSV.

**Exercise 4:** Create a 70/15/15 train/val/test split with a fixed seed and verify the sizes.

```python
from datasets import load_dataset

# Exercise 1
ds = load_dataset("glue", "mrpc")
print(ds["train"][:5])

# Exercise 4
split = ds["train"].train_test_split(test_size=0.3, seed=42)
train_val = split["train"].train_test_split(test_size=0.5, seed=42)
print(f"Train: {len(train_val['train'])}, Val: {len(train_val['test'])}, Test: {len(split['test'])}")
```


# Key Terms

---


| Term | What people say | What it actually means |
|------|----------------|----------------------|
| Dataset split | "Training data" | A named subset (train/val/test) used at different stages of the ML lifecycle |
| Streaming | "Load it lazily" | Processing data row by row from a remote source without downloading the full dataset |
| Parquet | "Compressed CSV" | A columnar file format optimized for analytical queries and storage efficiency |
| Arrow | "Fast dataframe" | An in-memory columnar format used internally by the datasets library for zero-copy reads |
| Git LFS | "Git for big files" | An extension that stores large files outside the git repo while keeping pointers in version control |
| DVC | "Git for data" | A version control system for datasets and models that integrates with cloud storage |
| Cache | "Already downloaded" | A local copy of previously fetched data, stored at ~/.cache/huggingface/ by default |


# Next Steps

---


When your data workflow is solid, continue to:
**Lesson 10: Terminal and Shell**

Your `ai-engineering-from-scratch` learning directory is **completely untouched** — these notes live separately in this folder.
